# STRAIN DREAMER.mat Model Training
This notebook demonstrates how the machine learning model is trained directly on the raw clinical `DREAMER.mat` brainwave dataset.

It loads the MATLAB structure, extracts the 14 EEG channels per trial, calculates energy proxies, and builds a regularized linear regression map predicting Valence (Positivity) and Arousal (Energy).

In [ ]:
from scipy.io import loadmat
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Load the raw MATLAB file (Requires data/raw/DREAMER.mat in your project)
print('Loading DREAMER.mat (this may take 30-60 seconds)...')
mat = loadmat('../data/raw/DREAMER.mat', squeeze_me=False, struct_as_record=False)
root = mat['DREAMER'][0, 0]
print('Successfully loaded DREAMER dataset!')

In [ ]:
# Extract features and labels
X_features = []
y_valence = []
y_arousal = []

n_subjects = root['Data'].shape[0] if root['Data'].ndim == 1 else max(root['Data'].shape)
print(f'Found {n_subjects} subjects in the dataset.')

# Loop through subjects (we will use all available subjects for a robust model)
for sub_id in range(n_subjects):
    sub = root['Data'][sub_id, 0] if root['Data'].ndim == 2 else root['Data'][sub_id]
    stimuli = sub['EEG'][0, 0]['stimuli'][0, 0]
    n_trials = int(stimuli.shape[0])
    
    for trial_id in range(n_trials):
        # Get EEG signal: Shape is (n_samples, 14 channels)
        sig = np.asarray(stimuli[trial_id, 0], dtype=np.float64)[:, :14]
        
        # Feature Engineering: Root Mean Square (RMS) energy for each of the 14 EEG channels across the whole trial
        features = np.sqrt(np.mean(sig**2, axis=0))
        X_features.append(features)
        
        # Get corresponding emotional labels (scale 1 to 5)
        v = float(sub['ScoreValence'][0, 0][trial_id, 0])
        a = float(sub['ScoreArousal'][0, 0][trial_id, 0])
        y_valence.append(v)
        y_arousal.append(a)

X = np.array(X_features)
y_v = np.array(y_valence)
y_a = np.array(y_arousal)
print(f'Extraction complete! Total extracted trials: {X.shape[0]}')

In [ ]:
# Train AI Models
X_train, X_test, y_v_train, y_v_test = train_test_split(X, y_v, test_size=0.2, random_state=42)
_, _, y_a_train, y_a_test = train_test_split(X, y_a, test_size=0.2, random_state=42)

# We use Ridge Regression (regularized linear model) paired with Standard Scaling
valence_model = make_pipeline(StandardScaler(), Ridge(alpha=100.0))
valence_model.fit(X_train, y_v_train)

arousal_model = make_pipeline(StandardScaler(), Ridge(alpha=100.0))
arousal_model.fit(X_train, y_a_train)

print('Valence Model test R^2 Score:', valence_model.score(X_test, y_v_test))
print('Arousal Model test R^2 Score:', arousal_model.score(X_test, y_a_test))
print('\nModels successfully trained on real DREAMER brainwave data!')